In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier as XGB
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from warnings import filterwarnings
filterwarnings('ignore')

In [ ]:
df = pd.read_csv('/content/kidney_disease.csv')

In [ ]:
df.head()

,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,...,38,6000,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,...,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,...,35,7300,4.6,no,no,no,good,no,no,ckd


In [ ]:
df['classification'] = df['classification'].str.strip()

In [ ]:
df['classification'].value_counts()

,count
classification,
ckd,250
notckd,150


In [ ]:
df['classification'] = df['classification'].map({'ckd': 1, 'notckd': 0})

In [ ]:
df.head()

,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,...,44,7800,5.2,yes,yes,no,good,no,no,1
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,...,38,6000,NaN,no,no,no,good,no,no,1
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,...,31,7500,NaN,no,yes,no,poor,no,yes,1
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,32,6700,3.9,yes,no,no,poor,yes,yes,1
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,...,35,7300,4.6,no,no,no,good,no,no,1


In [ ]:
df.isnull().sum()

,0
id,0
age,9
bp,12
sg,47
al,46
su,49
rbc,152
pc,65
pcc,4
ba,4


In [ ]:
df = df.drop('id', axis=1)

In [ ]:
df.shape

(400, 25)

In [ ]:
# categorical columns
cat_cols = ['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'ane', 'pe']
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Numeric columns → fill with median
numeric_cols = ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc',
                'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc']

for col in numeric_cols:
    # Replace '?' or '\t?' or anything non-numeric with NaN
    df[col] = df[col].replace(['?', '\t?', '\t43'], np.nan)
    # Convert to numeric (coerce errors → NaN)
    df[col] = pd.to_numeric(df[col], errors='coerce')
    # Fill missing with median
    df[col].fillna(df[col].median(), inplace=True)


In [ ]:
df.isnull().sum()

,0
age,0
bp,0
sg,0
al,0
su,0
rbc,0
pc,0
pcc,0
ba,0
bgr,0


In [ ]:
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

In [ ]:
df.head()

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,48.0,80.0,1.020,1.0,0.0,1,1,0,0,121.0,...,44.0,7800.0,5.2,1,4,1,0,0,0,1
1,7.0,50.0,1.020,4.0,0.0,1,1,0,0,121.0,...,38.0,6000.0,4.8,0,3,1,0,0,0,1
2,62.0,80.0,1.010,2.0,3.0,1,1,0,0,423.0,...,31.0,7500.0,4.8,0,4,1,1,0,1,1
3,48.0,70.0,1.005,4.0,0.0,1,0,1,0,117.0,...,32.0,6700.0,3.9,1,3,1,1,1,1,1
4,51.0,80.0,1.010,2.0,0.0,1,1,0,0,106.0,...,35.0,7300.0,4.6,0,3,1,0,0,0,1


In [ ]:
df.loc[300]

,300
age,45.00
bp,60.00
sg,1.02
al,0.00
su,0.00
rbc,1.00
pc,1.00
pcc,0.00
ba,0.00
bgr,114.00


In [ ]:
x = df.drop(['classification'], axis=1)
y = df['classification']

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, stratify=y ,test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [ ]:
smote = SMOTE()
x_train, y_train = smote.fit_resample(x_train, y_train)

In [ ]:
XGB(scale_pos_weight = (len(y_train[y_train==0]) / len(y_train[y_train==1])))

model = XGB(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.002,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(x_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.002, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
y_pred = model.predict(x_test)

In [ ]:
print("accuracy train data: ", accuracy_score(y_train, model.predict(x_train)))
print("accuracy test data: ", accuracy_score(y_test, y_pred))

accuracy train data:  0.99
accuracy test data:  0.9875


In [ ]:
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.97      0.98        30
           1       0.98      1.00      0.99        50

    accuracy                           0.99        80
   macro avg       0.99      0.98      0.99        80
weighted avg       0.99      0.99      0.99        80

[[29  1]
 [ 0 50]]


In [ ]:
# import pickle
# pickle.dump(model, open('model_kd.pkl', 'wb'))
# pickle.dump(scaler, open('scaler_kd.pkl', 'wb'))

In [ ]:
sample_input = [45.00,60.00,1.02,0.00,0.00,1.00,1.00,0.00,0.00,114.00,26.00,0.70,141.00,4.20,15.00,43.00,9200.00,5.80,0.00,3.00,1.00,0.00,0.00,0.00]
sample_input_arr = np.array(sample_input).reshape(1, -1)
sample_input_arr_scaled = scaler.transform(sample_input_arr)
prediction = model.predict(sample_input_arr_scaled)
print(prediction)

if prediction[0] == 0:
    print('Not CKD')
else:
    print('CKD')


[0]
Not CKD
